# LLM classification finetuning
First of all let's see sample of the data set

In [11]:
import numpy as np
import os
import pandas as pd
from xgboost import XGBClassifier


TRAIN_SET = pd.read_csv("/kaggle/input/competitions/llm-classification-finetuning/train.csv")
TRAIN_SET = TRAIN_SET.set_index("id")
TRAIN_SET2 = pd.DataFrame()

TEST_SET = pd.read_csv("/kaggle/input/competitions/llm-classification-finetuning/test.csv")
TEST_SET2 = pd.DataFrame()
print(TEST_SET.head())


        id                                             prompt  \
0   136060  ["I have three oranges today, I ate an orange ...   
1   211333  ["You are a mediator in a heated political deb...   
2  1233961  ["How to initialize the classification head wh...   

                                          response_a  \
0                    ["You have two oranges today."]   
1  ["Thank you for sharing the details of the sit...   
2  ["When you want to initialize the classificati...   

                                          response_b  
0  ["You still have three oranges. Eating an oran...  
1  ["Mr Reddy and Ms Blue both have valid points ...  
2  ["To initialize the classification head when p...  


Let's add some traits to make model easier to teach, but for now we need to add the column for final classification. I will add `who_won` column in which:
 - 0 is a `winner_tie` equals 1
 - 1 is a `winner_model_a` equals 1
 - 2 is a `winner_model_b` equals 1
`who_won` will be a y list for classification 

In [3]:
who_won = []
for amodel_bit, bmodel_bit, tie in zip(TRAIN_SET["winner_model_a"], TRAIN_SET["winner_model_b"],TRAIN_SET["winner_tie"]):
    if tie == 1:
        who_won.append(0)
    elif amodel_bit == 1:
        who_won.append(1)
    else:
        who_won.append(2)

## EDA
In this chapter we will do a bit of a EDA. First of all we will see if `model_a` and `model_b` are sets with the same models in them. It will make easier creating features, because user can prefer answers from 'better' models (like Claude or ChatGPT)

In [4]:
models_a = set(TRAIN_SET["model_a"])
models_b = set(TRAIN_SET["model_b"])

print(f"A models No. is {len(models_a)}")
print(f"B models No. is {len(models_b)}")

if models_a == models_b:
    print("These models ARE the same")
else:
    print("These models ARE NOT the same")

A models No. is 64
B models No. is 64
These models ARE the same


As we can see these models are the same which will make EDA more easier to implement. Plan for this competition is that I will create 2 model one of which is a transformer model for text and the second one is for classification for featured traits.

In [10]:
valsrefuse_list = ["as an ai", "i cannot", "i can't", "i refuse", "i'm sorry", "i apologize"]
refuses_counts_a = []
refuses_counts_b = []

for phrase in refuse_list:
    TRAIN_SET2[f"{phrase.replace(" ", "_")}_count_a"] = TRAIN_SET["response_a"].str.lower().str.count(phrase)
    TRAIN_SET2[f"{phrase.replace(" ", "_")}_count_b"] = TRAIN_SET["response_b"].str.lower().str.count(phrase)

len_diff = TRAIN_SET["response_a"].str.len() - TRAIN_SET["response_b"].str.len()
a_longer = len_diff > 0
the_same = len_diff == 0
b_longer = len_diff < 0
response_values = [0,1,2]
longer_response = np.select([the_same, a_longer, b_longer], response_values)
TRAIN_SET2["longer_response"] = longer_response

print(TRAIN_SET2.info())

<class 'pandas.core.frame.DataFrame'>
Index: 57477 entries, 30192 to 4294947231
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   as_an_ai_count_a     57477 non-null  int64
 1   as_an_ai_count_b     57477 non-null  int64
 2   i_cannot_count_a     57477 non-null  int64
 3   i_cannot_count_b     57477 non-null  int64
 4   i_can't_count_a      57477 non-null  int64
 5   i_can't_count_b      57477 non-null  int64
 6   i_refuse_count_a     57477 non-null  int64
 7   i_refuse_count_b     57477 non-null  int64
 8   i'm_sorry_count_a    57477 non-null  int64
 9   i'm_sorry_count_b    57477 non-null  int64
 10  i_apologize_count_a  57477 non-null  int64
 11  i_apologize_count_b  57477 non-null  int64
 12  as_ai_count_a        57477 non-null  int64
 13  as_ai_count_b        57477 non-null  int64
 14  longer_response      57477 non-null  int64
dtypes: int64(15)
memory usage: 7.0 MB
None


## Model
let's create our first demo model o predict which AI will prefer user based on first feature engineered traits